In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import gc
import pydicom as dicom
from scipy.ndimage import zoom



In [2]:
MIN_MAX = {
    "Weeks": (-5.0, 133.0),
    "FVC": (827.0, 6399.0),
    "Percent": (28.877577, 153.145378),
    "Age": (49.0, 88.0),
    "typical_fvc": (827.0, 6399.0),
}




In [3]:
def read_csv(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        alt_path = os.path.join("/kaggle/input", path.strip("../"))
        return pd.read_csv(alt_path)


TRAIN_DF = read_csv("../input/osic-pulmonary-fibrosis-progression/train.csv")
TEST_DF = read_csv("../input/osic-pulmonary-fibrosis-progression/test.csv")
SAMPLE_SUBMISSION = read_csv(
    "../input/osic-pulmonary-fibrosis-progression/sample_submission.csv"
)

global_slope, global_intercept = np.polyfit(TRAIN_DF["Weeks"], TRAIN_DF["FVC"], 1)

patient_slopes = {}
for pid, grp in TRAIN_DF.groupby("Patient"):
    if len(grp) > 1:
        slope_i, _ = np.polyfit(grp["Weeks"], grp["FVC"], 1)
        patient_slopes[pid] = slope_i

demographic_slopes = {}
for (sex, smoke), grp in TRAIN_DF.groupby(["Sex", "SmokingStatus"]):
    slopes = []
    for pid, subgrp in grp.groupby("Patient"):
        if pid in patient_slopes:
            slopes.append(patient_slopes[pid])
    if slopes:
        demographic_slopes[(sex, smoke)] = np.mean(slopes)




In [4]:
def create_typical_fvc(df):
    df["typical_fvc"] = df["FVC"] / df["Percent"] * 100.0
    return df


def normalize(df):
    for feature, min_max in MIN_MAX.items():
        if feature == "FVC":
            # skip normalising FVC – we need the raw value for baseline predictions
            continue
        df[feature] = (df[feature] - min_max[0]) / (min_max[1] - min_max[0])
    return df


TEST_DF = create_typical_fvc(TEST_DF)

# keep a copy of the original (raw) FVC before any scaling
TEST_DF["FVC_original"] = TEST_DF["FVC"]

TEST_DF["min_week"] = np.zeros(len(TEST_DF), dtype="int")
TEST_DF["min_week_FVC"] = np.zeros(len(TEST_DF), dtype="int")

TEST_DF["Never smoked"] = (TEST_DF["SmokingStatus"] == "Never smoked").astype("uint8")
TEST_DF["Currently smokes"] = (TEST_DF["SmokingStatus"] == "Currently smokes").astype(
    "uint8"
)
TEST_DF["Ex-smoker"] = (TEST_DF["SmokingStatus"] == "Ex-smoker").astype("uint8")
TEST_DF["Male"] = (TEST_DF["Sex"] == "Male").astype("uint8")
TEST_DF["Female"] = (TEST_DF["Sex"] == "Female").astype("uint8")

TEST_DF = normalize(TEST_DF)

for patient in np.unique(TEST_DF["Patient"]):
    mask = TEST_DF["Patient"] == patient
    TEST_DF.loc[mask, "min_week"] = TEST_DF.loc[mask, "Weeks"].min()
    # use the raw baseline FVC, not the (potentially altered) column
    TEST_DF.loc[mask, "min_week_FVC"] = TEST_DF.loc[mask, "FVC_original"].values[0]




/tmp/ipykernel_11/1320759521.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.036231884057971016' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  TEST_DF.loc[mask, "min_week"] = TEST_DF.loc[mask, "Weeks"].min()


In [5]:
# ---- compute a simple bias correction from training data ----
def _predict_from_df(df):
    preds = []
    for _, row in df.iterrows():
        patient = row["Patient"]
        week = row["Weeks"]
        # baseline for this patient in the same dataframe
        grp = df[df["Patient"] == patient]
        baseline_fvc = grp.iloc[0]["FVC"]
        baseline_week = grp["Weeks"].min()
        slope = patient_slopes.get(patient, None)
        if slope is None:
            demo_key = (grp.iloc[0]["Sex"], grp.iloc[0]["SmokingStatus"])
            slope = demographic_slopes.get(demo_key, global_slope)
        pred = baseline_fvc + slope * (week - baseline_week)
        pred = np.clip(pred, MIN_MAX["FVC"][0], MIN_MAX["FVC"][1])
        preds.append(pred)
    return np.array(preds)


train_preds = _predict_from_df(TRAIN_DF)
bias_correction = np.mean(TRAIN_DF["FVC"] - train_preds)  # average residual

pred_fvc = []
conf_vals = []

for idx, row in SAMPLE_SUBMISSION.iterrows():
    patient_id, week_str = row["Patient_Week"].split("_")
    week = int(week_str)

    patient_info = TEST_DF[TEST_DF["Patient"] == patient_id].iloc[0]
    baseline_fvc = patient_info["FVC_original"]  # raw baseline value
    baseline_week = patient_info["min_week"]

    slope_used = patient_slopes.get(patient_id, None)
    if slope_used is None:
        demo_key = (patient_info["Sex"], patient_info["SmokingStatus"])
        slope_used = demographic_slopes.get(demo_key, global_slope)

    fvc_pred = baseline_fvc + slope_used * (week - baseline_week)
    # apply bias correction learned from training data
    fvc_pred += bias_correction
    fvc_pred = np.clip(fvc_pred, MIN_MAX["FVC"][0], MIN_MAX["FVC"][1])

    pred_fvc.append(fvc_pred)
    conf_vals.append(200)  # keep confidence above the clipping threshold

SAMPLE_SUBMISSION["FVC"] = np.round(pred_fvc).astype(int)
SAMPLE_SUBMISSION["Confidence"] = np.array(conf_vals).astype(int)



In [6]:
submission_path = "submission.csv"
SAMPLE_SUBMISSION.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")
print(SAMPLE_SUBMISSION.head())

Submission written to submission.csv
                   Patient_Week   FVC  Confidence
0  ID00126637202218610655908_-3  2372         200
1  ID00126637202218610655908_-2  2367         200
2  ID00126637202218610655908_-1  2362         200
3   ID00126637202218610655908_0  2357         200
4   ID00126637202218610655908_1  2352         200
